In [2]:
import pandas as pd

#Archivos a utilizar para la revision con el listado del barco:
#-Concentrado2
concentrado2 = '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2026/Enero/Concentrado2_SoloEnero2026.xlsx'

#-FacturasFaltantes
facturasFaltantes = '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2026/Enero/Facturas_Faltantes_SoloEnero2026.xlsx'

#-ExtraccionHistorica
Extraccion = '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2026/Enero/ExtraccionGeneral2Enero2026.xlsx'
#-ChasisBarcoEJEMPLO
chasisBarco = '/Users/jorgevilchis/Downloads/chasis de barco SIEM CICERO 30.09.2025.xlsx'


dfConcentrado2 = pd.read_excel(concentrado2)
dfFacturasFaltantes = pd.read_excel(facturasFaltantes)
dfExtraccion = pd.read_excel(Extraccion)
dfChasisBarco = pd.read_excel(chasisBarco)
#Queremos ver que tenemos, que no, y las que no en donde es que estaban. 

print('Concentrado2:')
dfConcentrado2.head(10)
print('---'*20)
print('FacturasFaltantes:')
dfFacturasFaltantes.head(3)
print('---'*20)
print('ExtraccionHistorica:')
dfExtraccion.head(3)
print('---'*20) 
print('ChasisBarco:')
dfChasisBarco.head(3)
print('---'*20)


Concentrado2:
------------------------------------------------------------
FacturasFaltantes:
------------------------------------------------------------
ExtraccionHistorica:
------------------------------------------------------------
ChasisBarco:
------------------------------------------------------------


In [4]:
# === STEP 1: Load all boat manifest files ===
from pathlib import Path

boat_files = [
    '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2026/Enero/Facturas/Attachments-FW_ (VWKL) MV EMDEN V.029A - ATD BCN 30_12_25 - CARGO DOCUMENTATION BARCELONA/chasis de barco EMDEN 23.01.2026.xlsx',
    '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2025/Diciembre 2025/Facturas/SoloDiciembre/Attachments-FW_ Mexico Waybills _Wolfsburg_ 028A (1)/Chasis de barco WOLFSBURG 10.12.2025.xlsx',
    '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2025/Diciembre 2025/Facturas/Previas/fwmexicowaybillssiemconfucius071a/Chasis de barco SIEM CONFUCIUS 01.12.2025.xlsx',
    '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2025/Diciembre 2025/Facturas/Previas/fwmexicowaybillslakevictoria001a/Chasis de barco LAKE VICTORIA 03.11.2025.xlsx',
    '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2025/Octubre2025/FacturasPeriodoOctubre/fwmexicowaybillssiemcicero110a/chasis de barco SIEM CICERO 30.09.2025.xlsx',
    '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2025/Diciembre 2025/Facturas/Previas/fwmxwaybillsemden027a (1)/Chasis de barco EMDEN 14.11.2025.xlsx',
    '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2025/Agosto2025/Facturas/fwmexicowaybillswayforward010a_SinErrror/chasis de barco WAY FORWARD 12.08.2024.xlsx',
    '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2025/trashbuffer/fw20231233mexicohamburghighway031223/Chasis de barco HAMBURG HIGHWAY 28.12.2023.xlsx',
]

# Load and combine all boat manifests
all_boats = []
for f in boat_files:
    try:
        df = pd.read_excel(f)
        df['SourceFile'] = Path(f).name
        df['FolderPath'] = str(Path(f).parent)
        all_boats.append(df)
        print(f"[OK] Loaded: {Path(f).name} ({len(df)} rows)")
    except Exception as e:
        print(f"[ERROR] Loading {Path(f).name}: {e}")

dfAllBoats = pd.concat(all_boats, ignore_index=True)
print(f"\nTotal boat records: {len(dfAllBoats)}")
print(f"Unique boats: {dfAllBoats['Ship Name'].nunique()}")
dfAllBoats['Ship Name'].value_counts()

[OK] Loaded: chasis de barco EMDEN 23.01.2026.xlsx (1613 rows)
[OK] Loaded: Chasis de barco WOLFSBURG 10.12.2025.xlsx (122 rows)
[OK] Loaded: Chasis de barco SIEM CONFUCIUS 01.12.2025.xlsx (65 rows)
[OK] Loaded: Chasis de barco LAKE VICTORIA 03.11.2025.xlsx (70 rows)
[OK] Loaded: chasis de barco SIEM CICERO 30.09.2025.xlsx (98 rows)
[OK] Loaded: Chasis de barco EMDEN 14.11.2025.xlsx (965 rows)
[OK] Loaded: chasis de barco WAY FORWARD 12.08.2024.xlsx (1476 rows)
[OK] Loaded: Chasis de barco HAMBURG HIGHWAY 28.12.2023.xlsx (1050 rows)

Total boat records: 5459
Unique boats: 7


Ship Name
EMDEN              2578
WAY FORWARD        1476
HAMBURG HIGHWAY    1050
WOLFSBURG           122
SIEM CICERO          98
LAKE VICTORIA        70
SIEM CONFUCIUS       65
Name: count, dtype: int64

In [ ]:
# === STEP 2: Match FacturasFaltantes with boat data via Invoice Number ===

# Clean invoice columns for matching
dfFacturasFaltantes['FACT'] = dfFacturasFaltantes['FACT'].astype(str).str.strip()
dfAllBoats['FI: Invoice No.'] = dfAllBoats['FI: Invoice No.'].astype(str).str.strip()

# Match missing invoices to boats
dfFaltantesConBarco = dfFacturasFaltantes.merge(
    dfAllBoats[['FI: Invoice No.', 'Ship Name', 'Chassis Number', 'FolderPath']].drop_duplicates(),
    left_on='FACT',
    right_on='FI: Invoice No.',
    how='left'
)

# Count matches
matched = dfFaltantesConBarco['Ship Name'].notna().sum()
total = len(dfFaltantesConBarco)
print(f"Matched via Invoice: {matched}/{total} ({matched/total*100:.1f}%)")
print(f"\nBoats found in missing invoices:")
print(dfFaltantesConBarco[dfFaltantesConBarco['Ship Name'].notna()]['Ship Name'].value_counts())

NameError: name 'dfFacturasFaltantes' is not defined

In [5]:
# === STEP 3: For unmatched invoices, try matching via Chassis from dfExtraccion ===

# Get the invoices that weren't matched by invoice number
unmatched_invoices = dfFaltantesConBarco[dfFaltantesConBarco['Ship Name'].isna()]['FACT'].unique()
print(f"Trying to match {len(unmatched_invoices)} unmatched invoices via chassis...")

# Get chassis for these invoices from dfExtraccion
dfExtraccion['Factura'] = dfExtraccion['Factura'].astype(str).str.strip()
dfExtraccion['Chasis'] = dfExtraccion['Chasis'].astype(str).str.strip()
dfAllBoats['Chassis Number'] = dfAllBoats['Chassis Number'].astype(str).str.strip()

# Get unique invoice-chassis pairs from extraction
extraccion_subset = dfExtraccion[dfExtraccion['Factura'].isin(unmatched_invoices)][['Factura', 'Chasis']].drop_duplicates()

# Match these chassis against boat manifests
chassis_boat_match = extraccion_subset.merge(
    dfAllBoats[['Chassis Number', 'Ship Name', 'FolderPath']].drop_duplicates(),
    left_on='Chasis',
    right_on='Chassis Number',
    how='inner'
)

print(f"[OK] Found {len(chassis_boat_match)} additional matches via chassis")
if len(chassis_boat_match) > 0:
    print(f"\nAdditional boats found:")
    print(chassis_boat_match['Ship Name'].value_counts())

Trying to match 457 unmatched invoices via chassis...
[OK] Found 0 additional matches via chassis


In [6]:
# === STEP 4: Create Final Report ===

# Combine both matching methods
# First: direct invoice matches
direct_matches = dfFaltantesConBarco[dfFaltantesConBarco['Ship Name'].notna()][['FACT', 'Ship Name', 'FolderPath']].drop_duplicates()
direct_matches['Match Type'] = 'Invoice Number'

# Second: chassis matches (rename column)
chassis_matches = chassis_boat_match[['Factura', 'Ship Name', 'FolderPath']].drop_duplicates()
chassis_matches = chassis_matches.rename(columns={'Factura': 'FACT'})
chassis_matches['Match Type'] = 'Chassis'

# Combine
all_matches = pd.concat([direct_matches, chassis_matches], ignore_index=True).drop_duplicates(subset='FACT')

# Check which invoices exist in dfExtraccion (PDF was processed)
all_matches['Found in Extraccion'] = all_matches['FACT'].isin(dfExtraccion['Factura'].astype(str))

# Add the still-unmatched invoices
all_faltantes = dfFacturasFaltantes['FACT'].unique()
matched_faltantes = all_matches['FACT'].unique()
still_missing = [f for f in all_faltantes if f not in matched_faltantes]

# Create unmatched dataframe
unmatched_df = pd.DataFrame({'FACT': still_missing})
unmatched_df['Ship Name'] = 'NOT FOUND IN BOATS'
unmatched_df['FolderPath'] = 'UNKNOWN'
unmatched_df['Match Type'] = 'No Match'
unmatched_df['Found in Extraccion'] = unmatched_df['FACT'].isin(dfExtraccion['Factura'].astype(str))

# Final report
dfFinalReport = pd.concat([all_matches, unmatched_df], ignore_index=True)

print("=" * 60)
print("FINAL REPORT: FacturasFaltantes by Boat")
print("=" * 60)
print(f"\nTotal missing invoices: {len(dfFacturasFaltantes)}")
print(f"Matched to boats: {len(all_matches)}")
print(f"Not found in any boat: {len(still_missing)}")
print(f"\nBreakdown by Boat:")
print(dfFinalReport['Ship Name'].value_counts())
print(f"\nFiles found in extraction: {dfFinalReport['Found in Extraccion'].sum()}")
dfFinalReport.head(20)

FINAL REPORT: FacturasFaltantes by Boat

Total missing invoices: 458
Matched to boats: 1
Not found in any boat: 457

Breakdown by Boat:
Ship Name
NOT FOUND IN BOATS    457
SIEM CONFUCIUS          1
Name: count, dtype: int64

Files found in extraction: 1


,FACT,Ship Name,FolderPath,Match Type,Found in Extraccion
0,953325E,SIEM CONFUCIUS,/Users/jorgevilchis/Documents/Cosas Gestell/Ad...,Invoice Number,False
1,20523011,NOT FOUND IN BOATS,UNKNOWN,No Match,False
2,20523012,NOT FOUND IN BOATS,UNKNOWN,No Match,False
3,20523013,NOT FOUND IN BOATS,UNKNOWN,No Match,False
4,13017050,NOT FOUND IN BOATS,UNKNOWN,No Match,False
5,979617E,NOT FOUND IN BOATS,UNKNOWN,No Match,False
6,67073037,NOT FOUND IN BOATS,UNKNOWN,No Match,False
7,67073036,NOT FOUND IN BOATS,UNKNOWN,No Match,False
8,67073039,NOT FOUND IN BOATS,UNKNOWN,No Match,False
9,67073038,NOT FOUND IN BOATS,UNKNOWN,No Match,False


In [7]:
# === STEP 5: Export Report & Show Where to Find Files ===

# Save report to Excel
output_path = '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2026/Enero/FacturasFaltantes_ConBarco.xlsx'
dfFinalReport.to_excel(output_path, index=False)
print(f"Report saved to: {output_path}")

# Show unique folder paths where to find missing files
print("\n" + "=" * 60)
print("WHERE TO FIND THE MISSING FILES:")
print("=" * 60)

for boat, group in dfFinalReport[dfFinalReport['Ship Name'] != 'NOT FOUND IN BOATS'].groupby('Ship Name'):
    folder = group['FolderPath'].iloc[0]
    count = len(group)
    print(f"\n{boat} ({count} invoices):")
    print(f"   {folder}")

# Show invoices not found in any boat
not_in_boats = dfFinalReport[dfFinalReport['Ship Name'] == 'NOT FOUND IN BOATS']
if len(not_in_boats) > 0:
    print(f"\n[WARNING] {len(not_in_boats)} invoices NOT FOUND in any boat manifest:")
    print(not_in_boats['FACT'].tolist()[:20])
    if len(not_in_boats) > 20:
        print(f"   ... and {len(not_in_boats) - 20} more")

Report saved to: /Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2026/Enero/FacturasFaltantes_ConBarco.xlsx

WHERE TO FIND THE MISSING FILES:

SIEM CONFUCIUS (1 invoices):
   /Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2025/Diciembre 2025/Facturas/Previas/fwmexicowaybillssiemconfucius071a

[WARNING] 457 invoices NOT FOUND in any boat manifest:
['20523011', '20523012', '20523013', '13017050', '979617E', '67073037', '67073036', '67073039', '67073038', '67074953', '67074915', '67074907', '67074955', '67074906', '67074898', '67074954', '67074952', '67074941', '67074980', '67074964']
   ... and 437 more


In [8]:
# === FUNCTION: Find chassis from boats NOT in Extraccion ===

def find_missing_chassis_from_boats(dfBoats, dfExtraccion):
    """
    Finds chassis numbers from boat manifests that do not appear in the extraction.
    
    Parameters:
    - dfBoats: DataFrame with boat manifest data (must have 'Chassis Number' and 'Ship Name')
    - dfExtraccion: DataFrame with extraction data (must have 'Chasis')
    
    Returns:
    - DataFrame with missing chassis and their boat information
    """
    # Clean chassis columns
    boats_chassis = dfBoats['Chassis Number'].astype(str).str.strip().unique()
    extraccion_chassis = dfExtraccion['Chasis'].astype(str).str.strip().unique()
    
    # Find chassis in boats but NOT in extraccion
    missing_chassis = set(boats_chassis) - set(extraccion_chassis)
    
    # Get full info for missing chassis
    dfBoats_clean = dfBoats.copy()
    dfBoats_clean['Chassis Number'] = dfBoats_clean['Chassis Number'].astype(str).str.strip()
    
    dfMissing = dfBoats_clean[dfBoats_clean['Chassis Number'].isin(missing_chassis)].copy()
    
    print("=" * 60)
    print("CHASSIS FROM BOATS NOT FOUND IN EXTRACCION")
    print("=" * 60)
    print(f"\nTotal chassis in boat manifests: {len(boats_chassis)}")
    print(f"Total chassis in extraccion: {len(extraccion_chassis)}")
    print(f"Missing from extraccion: {len(missing_chassis)}")
    print(f"\nBreakdown by boat:")
    print(dfMissing['Ship Name'].value_counts())
    
    return dfMissing

# Run the function
dfMissingFromBoats = find_missing_chassis_from_boats(dfAllBoats, dfExtraccion)
dfMissingFromBoats.head(20)

CHASSIS FROM BOATS NOT FOUND IN EXTRACCION

Total chassis in boat manifests: 5459
Total chassis in extraccion: 149804
Missing from extraccion: 2

Breakdown by boat:
Ship Name
SIEM CONFUCIUS    2
Name: count, dtype: int64


,Chassis Number,Model,Ship Name,FI: Invoice No.,estatus,SourceFile,FolderPath,Estatus
1741,VSSBBAKP9T1019386,KP1CYS,SIEM CONFUCIUS,953325E,NaN,Chasis de barco SIEM CONFUCIUS 01.12.2025.xlsx,/Users/jorgevilchis/Documents/Cosas Gestell/Ad...,ok
1742,VSSAAAKP0T1019405,KP1BC5,SIEM CONFUCIUS,953344E,NaN,Chasis de barco SIEM CONFUCIUS 01.12.2025.xlsx,/Users/jorgevilchis/Documents/Cosas Gestell/Ad...,ok


In [9]:
# === FUNCTION: Check if invoices from missing chassis exist in Extraccion ===

def check_invoices_in_extraccion(dfMissing, dfExtraccion):
    """
    Checks if invoices associated with missing chassis exist in the extraction.
    Helps identify if the PDF was processed but chassis wasn't extracted,
    or if the entire invoice PDF is missing.
    
    Returns:
    - Tuple of (dfInvoiceFound, dfInvoiceNotFound)
    """
    dfMissing_clean = dfMissing.copy()
    dfMissing_clean['FI: Invoice No.'] = dfMissing_clean['FI: Invoice No.'].astype(str).str.strip()
    
    extraccion_invoices = dfExtraccion['Factura'].astype(str).str.strip().unique()
    
    dfMissing_clean['Invoice in Extraccion'] = dfMissing_clean['FI: Invoice No.'].isin(extraccion_invoices)
    
    dfInvoiceFound = dfMissing_clean[dfMissing_clean['Invoice in Extraccion'] == True].copy()
    dfInvoiceNotFound = dfMissing_clean[dfMissing_clean['Invoice in Extraccion'] == False].copy()
    
    print("=" * 60)
    print("INVOICE STATUS FOR MISSING CHASSIS")
    print("=" * 60)
    print(f"\nTotal missing chassis records: {len(dfMissing_clean)}")
    print(f"Unique invoices in missing chassis: {dfMissing_clean['FI: Invoice No.'].nunique()}")
    print(f"\nInvoices FOUND in extraccion: {dfInvoiceFound['FI: Invoice No.'].nunique()}")
    print("  -> Chassis missing but invoice PDF was processed")
    print(f"\nInvoices NOT FOUND in extraccion: {dfInvoiceNotFound['FI: Invoice No.'].nunique()}")
    print("  -> Entire invoice PDF is missing")
    
    if len(dfInvoiceNotFound) > 0:
        print(f"\nMissing invoices by boat:")
        print(dfInvoiceNotFound.groupby('Ship Name')['FI: Invoice No.'].nunique())
    
    return dfInvoiceFound, dfInvoiceNotFound

# Run the function
dfChassisFound, dfChassisNotFound = check_invoices_in_extraccion(dfMissingFromBoats, dfExtraccion)

# Show invoices that need to be located
print("\n" + "=" * 60)
print("INVOICES TO LOCATE (PDF not processed):")
print("=" * 60)
if len(dfChassisNotFound) > 0:
    missing_invoices = dfChassisNotFound[['FI: Invoice No.', 'Ship Name', 'FolderPath']].drop_duplicates()
    print(missing_invoices)


INVOICE STATUS FOR MISSING CHASSIS

Total missing chassis records: 2
Unique invoices in missing chassis: 2

Invoices FOUND in extraccion: 0
  -> Chassis missing but invoice PDF was processed

Invoices NOT FOUND in extraccion: 2
  -> Entire invoice PDF is missing

Missing invoices by boat:
Ship Name
SIEM CONFUCIUS    2
Name: FI: Invoice No., dtype: int64

INVOICES TO LOCATE (PDF not processed):
     FI: Invoice No.       Ship Name  \
1741         953325E  SIEM CONFUCIUS   
1742         953344E  SIEM CONFUCIUS   

                                             FolderPath  
1741  /Users/jorgevilchis/Documents/Cosas Gestell/Ad...  
1742  /Users/jorgevilchis/Documents/Cosas Gestell/Ad...  


In [10]:
# === FUNCTION: Check FacturasFaltantes against Extraccion ===

def check_facturas_faltantes_in_extraccion(dfFaltantes, dfExtraccion):
    """
    Checks if the missing invoices (FacturasFaltantes) exist in the extraction.
    
    Returns:
    - Tuple of (dfFound, dfNotFound)
    """
    dfFaltantes_clean = dfFaltantes.copy()
    dfFaltantes_clean['FACT'] = dfFaltantes_clean['FACT'].astype(str).str.strip()
    
    extraccion_invoices = dfExtraccion['Factura'].astype(str).str.strip().unique()
    
    dfFaltantes_clean['In Extraccion'] = dfFaltantes_clean['FACT'].isin(extraccion_invoices)
    
    dfFound = dfFaltantes_clean[dfFaltantes_clean['In Extraccion'] == True].copy()
    dfNotFound = dfFaltantes_clean[dfFaltantes_clean['In Extraccion'] == False].copy()
    
    print("=" * 60)
    print("FACTURAS FALTANTES vs EXTRACCION")
    print("=" * 60)
    print(f"\nTotal FacturasFaltantes: {len(dfFaltantes_clean)}")
    print(f"\nFOUND in extraccion: {len(dfFound)}")
    print("  -> PDF was processed, J y N should be available")
    print(f"\nNOT FOUND in extraccion: {len(dfNotFound)}")
    print("  -> PDF was never processed, need to locate files")
    
    return dfFound, dfNotFound

# Run the function
dfFaltantesFound, dfFaltantesNotFound = check_facturas_faltantes_in_extraccion(dfFacturasFaltantes, dfExtraccion)

print("\n" + "=" * 60)
print("FACTURAS FALTANTES NOT IN EXTRACCION:")
print("=" * 60)
print(dfFaltantesNotFound['FACT'].tolist()[:30])
if len(dfFaltantesNotFound) > 30:
    print(f"... and {len(dfFaltantesNotFound) - 30} more")

FACTURAS FALTANTES vs EXTRACCION

Total FacturasFaltantes: 458

FOUND in extraccion: 1
  -> PDF was processed, J y N should be available

NOT FOUND in extraccion: 457
  -> PDF was never processed, need to locate files

FACTURAS FALTANTES NOT IN EXTRACCION:
['20523011', '20523012', '20523013', '13017050', '979617E', '67073037', '67073036', '67073039', '67073038', '67074953', '67074915', '67074907', '67074955', '67074906', '67074898', '67074954', '67074952', '67074941', '67074980', '67074964', '67074899', '67074864', '67075015', '67075030', '67075067', '67074987', '67075050', '67075044', '67075092', '67075037']
... and 427 more


In [11]:
# === FUNCTION: Check FacturasFaltantes against Boat Manifests ===

def check_facturas_faltantes_in_boats(dfFaltantes, dfBoats):
    """
    Checks if the missing invoices (FacturasFaltantes) appear in boat manifests.
    
    Returns:
    - Tuple of (dfFoundInBoats, dfNotFoundInBoats)
    """
    dfFaltantes_clean = dfFaltantes.copy()
    dfFaltantes_clean['FACT'] = dfFaltantes_clean['FACT'].astype(str).str.strip()
    
    boat_invoices = dfBoats['FI: Invoice No.'].astype(str).str.strip().unique()
    
    dfFaltantes_clean['In Boats'] = dfFaltantes_clean['FACT'].isin(boat_invoices)
    
    dfFoundInBoats = dfFaltantes_clean[dfFaltantes_clean['In Boats'] == True].copy()
    dfNotFoundInBoats = dfFaltantes_clean[dfFaltantes_clean['In Boats'] == False].copy()
    
    # Get boat info for found invoices
    if len(dfFoundInBoats) > 0:
        dfFoundInBoats = dfFoundInBoats.merge(
            dfBoats[['FI: Invoice No.', 'Ship Name', 'FolderPath']].drop_duplicates(),
            left_on='FACT',
            right_on='FI: Invoice No.',
            how='left'
        )
    
    print("=" * 60)
    print("FACTURAS FALTANTES vs BOAT MANIFESTS")
    print("=" * 60)
    print(f"\nTotal FacturasFaltantes: {len(dfFaltantes_clean)}")
    print(f"\nFOUND in boat manifests: {len(dfFoundInBoats)}")
    if len(dfFoundInBoats) > 0:
        print("  Breakdown by boat:")
        print(dfFoundInBoats['Ship Name'].value_counts())
    print(f"\nNOT FOUND in any boat manifest: {len(dfNotFoundInBoats)}")
    
    return dfFoundInBoats, dfNotFoundInBoats

# Run the function
dfFaltantesInBoats, dfFaltantesNotInBoats = check_facturas_faltantes_in_boats(dfFacturasFaltantes, dfAllBoats)

# Show where to find the invoices that ARE in boats
if len(dfFaltantesInBoats) > 0:
    print("\n" + "=" * 60)
    print("WHERE TO FIND THESE INVOICES:")
    print("=" * 60)
    for boat, group in dfFaltantesInBoats.groupby('Ship Name'):
        folder = group['FolderPath'].iloc[0]
        invoices = group['FACT'].unique()
        print(f"\n{boat} ({len(invoices)} invoices):")
        print(f"   Folder: {folder}")
        print(f"   Invoices: {list(invoices)[:10]}")
        if len(invoices) > 10:
            print(f"   ... and {len(invoices) - 10} more")

FACTURAS FALTANTES vs BOAT MANIFESTS

Total FacturasFaltantes: 458

FOUND in boat manifests: 1
  Breakdown by boat:
Ship Name
SIEM CONFUCIUS    1
Name: count, dtype: int64

NOT FOUND in any boat manifest: 457

WHERE TO FIND THESE INVOICES:

SIEM CONFUCIUS (1 invoices):
   Folder: /Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2025/Diciembre 2025/Facturas/Previas/fwmexicowaybillssiemconfucius071a
   Invoices: ['953325E']


In [16]:
dfAllBoats.columns

Index(['Chassis Number', 'Model', 'Ship Name', 'FI: Invoice No.', 'estatus',
       'SourceFile', 'FolderPath', 'Estatus'],
      dtype='object')

In [19]:
from boat_analysis import *
# === STEP 1: Load all boat manifest files ===
from pathlib import Path

boat_files = [
    '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2026/Enero/Facturas/Attachments-FW_ (VWKL) MV EMDEN V.029A - ATD BCN 30_12_25 - CARGO DOCUMENTATION BARCELONA/chasis de barco EMDEN 23.01.2026.xlsx',
    '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2025/Diciembre 2025/Facturas/SoloDiciembre/Attachments-FW_ Mexico Waybills _Wolfsburg_ 028A (1)/Chasis de barco WOLFSBURG 10.12.2025.xlsx',
    '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2025/Diciembre 2025/Facturas/Previas/fwmexicowaybillssiemconfucius071a/Chasis de barco SIEM CONFUCIUS 01.12.2025.xlsx',
    '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2025/Diciembre 2025/Facturas/Previas/fwmexicowaybillslakevictoria001a/Chasis de barco LAKE VICTORIA 03.11.2025.xlsx',
    '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2025/Octubre2025/FacturasPeriodoOctubre/fwmexicowaybillssiemcicero110a/chasis de barco SIEM CICERO 30.09.2025.xlsx',
    '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2025/Diciembre 2025/Facturas/Previas/fwmxwaybillsemden027a (1)/Chasis de barco EMDEN 14.11.2025.xlsx',
    '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2025/Agosto2025/Facturas/fwmexicowaybillswayforward010a_SinErrror/chasis de barco WAY FORWARD 12.08.2024.xlsx',
    '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2025/trashbuffer/fw20231233mexicohamburghighway031223/Chasis de barco HAMBURG HIGHWAY 28.12.2023.xlsx',
]

#Archivos a utilizar para la revision con el listado del barco:
#-Concentrado2
concentrado2 = '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2026/Enero/Concentrado2_SoloEnero2026.xlsx'

#-FacturasFaltantes
facturasFaltantes = '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2026/Enero/Facturas_Faltantes_SoloEnero2026.xlsx'

#-ExtraccionHistorica
Extraccion = '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2026/Enero/ExtraccionGeneral2Enero2026.xlsx'
#-ChasisBarcoEJEMPLO
chasisBarco = '/Users/jorgevilchis/Downloads/chasis de barco SIEM CICERO 30.09.2025.xlsx'


dfConcentrado2 = pd.read_excel(concentrado2)
dfFacturasFaltantes = pd.read_excel(facturasFaltantes)
dfExtraccion = pd.read_excel(Extraccion)
dfChasisBarco = pd.read_excel(chasisBarco)
#Queremos ver que tenemos, que no, y las que no en donde es que estaban. 


# Load and combine all boat manifests
all_boats = []
for f in boat_files:
    try:
        df = pd.read_excel(f)
        df['SourceFile'] = Path(f).name
        df['FolderPath'] = str(Path(f).parent)
        all_boats.append(df)
        print(f"[OK] Loaded: {Path(f).name} ({len(df)} rows)")
    except Exception as e:
        print(f"[ERROR] Loading {Path(f).name}: {e}")

dfAllBoats = pd.concat(all_boats, ignore_index=True)
print(f"\nTotal boat records: {len(dfAllBoats)}")
print(f"Unique boats: {dfAllBoats['Ship Name'].nunique()}")
dfAllBoats['Ship Name'].value_counts()
# Load data
df_boats = load_boat_manifests(boat_files)

# Run everything
results = run_full_analysis(df_boats, dfExtraccion, dfFacturasFaltantes)


[OK] Loaded: chasis de barco EMDEN 23.01.2026.xlsx (1613 rows)
[OK] Loaded: Chasis de barco WOLFSBURG 10.12.2025.xlsx (122 rows)
[OK] Loaded: Chasis de barco SIEM CONFUCIUS 01.12.2025.xlsx (65 rows)
[OK] Loaded: Chasis de barco LAKE VICTORIA 03.11.2025.xlsx (70 rows)
[OK] Loaded: chasis de barco SIEM CICERO 30.09.2025.xlsx (98 rows)
[OK] Loaded: Chasis de barco EMDEN 14.11.2025.xlsx (965 rows)
[OK] Loaded: chasis de barco WAY FORWARD 12.08.2024.xlsx (1476 rows)
[OK] Loaded: Chasis de barco HAMBURG HIGHWAY 28.12.2023.xlsx (1050 rows)

Total boat records: 5459
Unique boats: 7
[OK] chasis de barco EMDEN 23.01.2026.xlsx (1613 rows)
[OK] Chasis de barco WOLFSBURG 10.12.2025.xlsx (122 rows)
[OK] Chasis de barco SIEM CONFUCIUS 01.12.2025.xlsx (65 rows)
[OK] Chasis de barco LAKE VICTORIA 03.11.2025.xlsx (70 rows)
[OK] chasis de barco SIEM CICERO 30.09.2025.xlsx (98 rows)
[OK] Chasis de barco EMDEN 14.11.2025.xlsx (965 rows)
[OK] chasis de barco WAY FORWARD 12.08.2024.xlsx (1476 rows)
[OK] Chas

In [ ]:
import importlib
import boat_analysis
importlib.reload(boat_analysis)
from boat_analysis import *

results = run_full_analysis(dfAllBoats, dfExtraccion, dfFacturasFaltantes)